# NETI-HyOptima Phase 2: Monte Carlo Simulation

This notebook demonstrates the Monte Carlo simulation layer for uncertainty analysis.

## Overview

Phase 2 adds stochastic capabilities to the HyOptima optimization engine:
- Sample from uncertainty distributions (solar CF, fuel cost, demand)
- Run multiple optimization scenarios
- Compute statistical distributions of outcomes
- Calculate risk metrics (VaR, CVaR)

## Uncertainties Modeled

1. **Solar Capacity Factor**: Weather variability (±15%)
2. **Fuel Cost**: Price volatility (triangular distribution)
3. **Demand**: Forecast uncertainty (±10%)

---

## 1. Setup and Imports

In [ ]:
# Standard imports
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

# HyOptima imports
from hyoptima import (
    HyOptimaModel,
    HyOptimaSolver,
    EconomicParameters,
    TechnicalParameters,
    LoadProfile,
    SolarProfile,
    # Simulation module
    MonteCarloSimulator,
    UncertaintyDistribution,
    SimulationConfig,
    create_default_uncertainties,
)

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("NETI-HyOptima Phase 2: Monte Carlo Simulation")
print("="*50)

## 2. Define Base Scenario

In [ ]:
# Create base load and solar profiles
load_profile = LoadProfile.generate_synthetic(
    peak_demand=200,      # 200 kW peak
    profile_type="mixed",
    hours=24,
    name="base_load"
)

solar_profile = SolarProfile.from_capacity_factor(
    capacity_factor=0.20,  # 20% average CF
    hours=24
)

# Economic parameters
economic_params = EconomicParameters(
    solar_capex=800,
    gas_capex=500,
    battery_capex=300,
    gas_fuel_cost=0.08,
    grid_tariff=0.12,
    unserved_energy_penalty=1.0,
    discount_rate=0.10,
)

print("Base Scenario:")
print(f"  Peak Demand: {load_profile.peak_demand:.0f} kW")
print(f"  Total Energy: {load_profile.total_energy:.0f} kWh")
print(f"  Solar CF: 20%")

## 3. Define Uncertainty Distributions

In [ ]:
# Use default uncertainties for Nigerian energy system
uncertainties = create_default_uncertainties()

print("Uncertainty Distributions:")
print("-" * 50)
for u in uncertainties:
    print(f"\n{u.name}:")
    print(f"  Base Value: {u.base_value}")
    print(f"  Distribution: {u.distribution_type}")
    if u.distribution_type == "normal":
        print(f"  Std Dev: {u.std_dev}")
    elif u.distribution_type == "triangular":
        print(f"  Min: {u.min_value}, Mode: {u.mode}, Max: {u.max_value}")

## 4. Visualize Uncertainty Distributions

In [ ]:
# Sample and visualize each distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

np.random.seed(42)
n_samples = 1000

for i, u in enumerate(uncertainties):
    samples = u.sample(n_samples)
    
    axes[i].hist(samples, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
    axes[i].axvline(u.base_value, color='red', linestyle='--', linewidth=2, label=f'Base: {u.base_value}')
    axes[i].set_xlabel(u.name)
    axes[i].set_ylabel('Frequency')
    axes[i].set_title(f'{u.name} Distribution\n({u.distribution_type})')
    axes[i].legend()

plt.tight_layout()
plt.savefig('../results/figures/uncertainty_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Run Monte Carlo Simulation

In [ ]:
# Configure simulation
config = SimulationConfig(
    n_scenarios=100,    # 100 scenarios
    seed=42,            # Reproducibility
    n_workers=0,        # Sequential (use -1 for parallel)
    verbose=True
)

# Create simulator
simulator = MonteCarloSimulator(uncertainties, config)

# Run simulation
print("\nRunning Monte Carlo Simulation...")
results = simulator.run(
    load_profile=load_profile,
    solar_profile=solar_profile,
    economic_params=economic_params
)

print(f"\nCompleted {len(results)} scenarios successfully")

## 6. Analyze Results

In [ ]:
# Get summary statistics
summary = simulator.summary()

print("="*60)
print("MONTE CARLO SIMULATION RESULTS")
print("="*60)

print("\n--- Capacity Decisions ---")
for key in ['solar_capacity_kw', 'gas_capacity_kw', 'battery_capacity_kwh']:
    s = summary[key]
    print(f"{key}:")
    print(f"  Mean: {s['mean']:.1f} ± {s['std']:.1f}")
    print(f"  Range: [{s['min']:.1f}, {s['max']:.1f}]")
    print(f"  90% CI: [{s['p5']:.1f}, {s['p95']:.1f}]")

print("\n--- Cost ---")
s = summary['total_cost']
print(f"Total Cost:")
print(f"  Mean: ${s['mean']:.2f} ± ${s['std']:.2f}")
print(f"  Range: [${s['min']:.2f}, ${s['max']:.2f}]")
print(f"  90% CI: [${s['p5']:.2f}, ${s['p95']:.2f}]")

In [ ]:
# Get risk metrics
risk = simulator.risk_metrics()

print("\n--- Risk Metrics ---")
print(f"Value at Risk (95%): ${risk['var_95']:.2f}")
print(f"Conditional VaR (95%): ${risk['cvar_95']:.2f}")
print(f"Worst Case: ${risk['worst_case']:.2f}")
print(f"Best Case: ${risk['best_case']:.2f}")
print(f"Coefficient of Variation: {risk['coefficient_of_variation']:.1%}")

## 7. Visualize Distributions

In [ ]:
# Convert to DataFrame for easier analysis
df = simulator.to_dataframe()

# Plot distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Solar capacity distribution
axes[0, 0].hist(df['solar_capacity_kw'], bins=30, alpha=0.7, color='#FFD700', edgecolor='white')
axes[0, 0].axvline(summary['solar_capacity_kw']['mean'], color='red', linestyle='--', 
                   label=f"Mean: {summary['solar_capacity_kw']['mean']:.0f} kW")
axes[0, 0].set_xlabel('Solar Capacity (kW)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Solar Capacity Distribution')
axes[0, 0].legend()

# Gas capacity distribution
axes[0, 1].hist(df['gas_capacity_kw'], bins=30, alpha=0.7, color='#8B4513', edgecolor='white')
axes[0, 1].axvline(summary['gas_capacity_kw']['mean'], color='red', linestyle='--',
                   label=f"Mean: {summary['gas_capacity_kw']['mean']:.0f} kW")
axes[0, 1].set_xlabel('Gas Capacity (kW)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Gas Capacity Distribution')
axes[0, 1].legend()

# Battery capacity distribution
axes[1, 0].hist(df['battery_capacity_kwh'], bins=30, alpha=0.7, color='#228B22', edgecolor='white')
axes[1, 0].axvline(summary['battery_capacity_kwh']['mean'], color='red', linestyle='--',
                   label=f"Mean: {summary['battery_capacity_kwh']['mean']:.0f} kWh")
axes[1, 0].set_xlabel('Battery Capacity (kWh)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Battery Capacity Distribution')
axes[1, 0].legend()

# Total cost distribution with risk metrics
axes[1, 1].hist(df['total_cost'], bins=30, alpha=0.7, color='#2196F3', edgecolor='white')
axes[1, 1].axvline(risk['var_95'], color='orange', linestyle='--', linewidth=2,
                   label=f"VaR 95%: ${risk['var_95']:.0f}")
axes[1, 1].axvline(risk['cvar_95'], color='red', linestyle='--', linewidth=2,
                   label=f"CVaR 95%: ${risk['cvar_95']:.0f}")
axes[1, 1].axvline(summary['total_cost']['mean'], color='green', linestyle='--',
                   label=f"Mean: ${summary['total_cost']['mean']:.0f}")
axes[1, 1].set_xlabel('Total Cost ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Total Cost Distribution with Risk Metrics')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../results/figures/monte_carlo_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlation Analysis

In [ ]:
# Analyze correlations between inputs and outputs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Solar CF vs Solar Capacity
axes[0].scatter(df['solar_cf'], df['solar_capacity_kw'], alpha=0.5, c='#FFD700')
axes[0].set_xlabel('Sampled Solar CF')
axes[0].set_ylabel('Optimal Solar Capacity (kW)')
axes[0].set_title('Solar CF vs Solar Capacity')
# Add trend line
z = np.polyfit(df['solar_cf'], df['solar_capacity_kw'], 1)
p = np.poly1d(z)
axes[0].plot(df['solar_cf'].sort_values(), p(df['solar_cf'].sort_values()), 
             'r--', linewidth=2, label=f'Trend: {z[0]:.0f} kW per 0.01 CF')
axes[0].legend()

# Fuel Cost vs Gas Capacity
axes[1].scatter(df['fuel_cost'], df['gas_capacity_kw'], alpha=0.5, c='#8B4513')
axes[1].set_xlabel('Sampled Fuel Cost ($/kWh)')
axes[1].set_ylabel('Optimal Gas Capacity (kW)')
axes[1].set_title('Fuel Cost vs Gas Capacity')
z = np.polyfit(df['fuel_cost'], df['gas_capacity_kw'], 1)
p = np.poly1d(z)
axes[1].plot(df['fuel_cost'].sort_values(), p(df['fuel_cost'].sort_values()), 
             'r--', linewidth=2, label=f'Trend: {z[0]:.0f} kW per $0.01/kWh')
axes[1].legend()

# Demand Multiplier vs Total Cost
axes[2].scatter(df['demand_multiplier'], df['total_cost'], alpha=0.5, c='#2196F3')
axes[2].set_xlabel('Demand Multiplier')
axes[2].set_ylabel('Total Cost ($)')
axes[2].set_title('Demand vs Total Cost')
z = np.polyfit(df['demand_multiplier'], df['total_cost'], 1)
p = np.poly1d(z)
axes[2].plot(df['demand_multiplier'].sort_values(), p(df['demand_multiplier'].sort_values()), 
             'r--', linewidth=2, label=f'Trend: ${z[0]:.0f} per 1% demand')
axes[2].legend()

plt.tight_layout()
plt.savefig('../results/figures/monte_carlo_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Scenario Analysis: Best vs Worst Case

In [ ]:
# Find best and worst case scenarios
best_idx = df['total_cost'].idxmin()
worst_idx = df['total_cost'].idxmax()

best = df.loc[best_idx]
worst = df.loc[worst_idx]

print("="*60)
print("SCENARIO ANALYSIS: BEST vs WORST CASE")
print("="*60)

print("\n--- Best Case Scenario ---")
print(f"Total Cost: ${best['total_cost']:.2f}")
print(f"Solar Capacity: {best['solar_capacity_kw']:.1f} kW")
print(f"Gas Capacity: {best['gas_capacity_kw']:.1f} kW")
print(f"Battery Capacity: {best['battery_capacity_kwh']:.1f} kWh")
print(f"\nSampled Parameters:")
print(f"  Solar CF: {best['solar_cf']:.1%}")
print(f"  Fuel Cost: ${best['fuel_cost']:.3f}/kWh")
print(f"  Demand Multiplier: {best['demand_multiplier']:.1%}")

print("\n--- Worst Case Scenario ---")
print(f"Total Cost: ${worst['total_cost']:.2f}")
print(f"Solar Capacity: {worst['solar_capacity_kw']:.1f} kW")
print(f"Gas Capacity: {worst['gas_capacity_kw']:.1f} kW")
print(f"Battery Capacity: {worst['battery_capacity_kwh']:.1f} kWh")
print(f"\nSampled Parameters:")
print(f"  Solar CF: {worst['solar_cf']:.1%}")
print(f"  Fuel Cost: ${worst['fuel_cost']:.3f}/kWh")
print(f"  Demand Multiplier: {worst['demand_multiplier']:.1%}")

print("\n--- Cost Difference ---")
print(f"Range: ${worst['total_cost'] - best['total_cost']:.2f}")
print(f"As % of Mean: {((worst['total_cost'] - best['total_cost']) / summary['total_cost']['mean']) * 100:.1f}%")

## 10. Save Results

In [ ]:
# Save simulation results
simulator.save_results('../results/runs/phase2_simulation_results.json')

# Also save DataFrame as CSV
df.to_csv('../results/runs/phase2_scenarios.csv', index=False)

print("Results saved to:")
print("  - ../results/runs/phase2_simulation_results.json")
print("  - ../results/runs/phase2_scenarios.csv")
print("  - ../results/figures/monte_carlo_distributions.png")
print("  - ../results/figures/monte_carlo_correlations.png")

## 11. Summary

In [ ]:
print("\n" + "="*70)
print("NETI-HyOptima Phase 2: Monte Carlo Simulation Summary")
print("="*70)

print(f"""
Key Findings:

1. CAPACITY UNCERTAINTY
   - Solar capacity varies {summary['solar_capacity_kw']['std']:.0f} kW around mean
   - Gas capacity varies {summary['gas_capacity_kw']['std']:.0f} kW around mean
   - Battery capacity varies {summary['battery_capacity_kwh']['std']:.0f} kWh around mean

2. COST RISK
   - Expected cost: ${summary['total_cost']['mean']:.2f}
   - 95% VaR: ${risk['var_95']:.2f} (5% chance of exceeding)
   - 95% CVaR: ${risk['cvar_95']:.2f} (expected cost in worst 5%)

3. KEY DRIVERS
   - Solar CF strongly influences solar capacity decision
   - Fuel cost affects gas/solar trade-off
   - Demand uncertainty drives overall cost variability

4. DECISION INSIGHTS
   - Robust system design should account for parameter uncertainty
   - Risk-averse planners should consider CVaR, not just expected cost
   - Sensitivity to fuel cost suggests hedging strategies

NEXT STEPS (Phase 3):
- Data engineering pipelines for real Nigerian data
- Backend API development
- Frontend dashboard
""")

print("="*70)
print("Phase 2 Complete: Monte Carlo simulation layer operational")
print("="*70)